# NYC Building Permits ETL Pipeline
## Transform Phase

Raw permit data is transformed using PySpark to produce a 
clean, analytics-ready dataset. Transformations include 
schema enforcement, column standardization, missing value 
handling, and derived field creation.

**Input:** Raw CSV checkpoint from Extract phase
**Engine:** Apache PySpark (local mode)
**Output:** Cleaned Parquet file ready for loading

#### Import Libraries
Libraries for distributed data processing, schema definition, 
and environment variable management.

In [1]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully")

Libraries loaded successfully


#### Initialize PySpark Session
A local SparkSession is initialized as the processing engine 
for all downstream transformations.

In [2]:
# Initialize PySpark session
spark = SparkSession.builder \
    .appName("NYC Permits ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("Spark session initialized successfully!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/22 15:56:40 WARN Utils: Your hostname, Catalinas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.16 instead (on interface en0)
26/03/22 15:56:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 15:56:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark session initialized successfully!


#### Load Raw Data into Spark
The raw CSV checkpoint from the Extract phase is loaded 
into a Spark DataFrame for distributed processing.

In [3]:
# Load raw CSV with all columns as strings
df = spark.read.csv(
    "/Users/catalinavalencia/Documents/ETL-ELT Project/data/raw_permits.csv",
    header=True,
    inferSchema=False  # Read everything as string first
)

# Clean column names
import re
def clean_column_name(name):
    name = name.lower()
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    return name

new_columns = [clean_column_name(c) for c in df.columns]
df = df.toDF(*new_columns)

# Cast types safely
df = df.withColumn("filing_date", F.expr("try_to_timestamp(filing_date, 'MM/dd/yyyy')")) \
       .withColumn("issuance_date", F.expr("try_to_timestamp(issuance_date, 'MM/dd/yyyy')")) \
       .withColumn("expiration_date", F.expr("try_to_timestamp(expiration_date, 'MM/dd/yyyy')"))

print("Schema:")
df.printSchema()
print(f"Row count: {df.count()}")

Schema:
root
 |-- borough: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- block: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- permit_type: string (nullable = true)
 |-- permit_status: string (nullable = true)
 |-- filing_date: timestamp (nullable = true)
 |-- issuance_date: timestamp (nullable = true)
 |-- expiration_date: timestamp (nullable = true)
 |-- work_type: string (nullable = true)
 |-- residential: string (nullable = true)
 |-- bldg_type: string (nullable = true)
 |-- owner_s_business_name: string (nullable = true)
 |-- owner_s_business_type: string (nullable = true)
 |-- permittee_s_business_name: string (nullable = true)
 |-- permittee_s_license_type: string (nullable = true)
 |-- gis_latitude: string (nullable = true)
 |-- gis_longitude: string (nullable = true)
 |-- gis_nta_name: string (nullable = true)

Row count: 50000


#### Enforce Data Types & Handle Missing Values
Numeric fields are cast to appropriate types and null values 
are replaced with standardized defaults to ensure downstream 
query reliability without distorting analysis.

In [4]:
# Cast numeric columns
df = df.withColumn("gis_latitude", F.col("gis_latitude").cast("double")) \
       .withColumn("gis_longitude", F.col("gis_longitude").cast("double"))

# Cast date columns
df = df.withColumn("filing_date", F.expr("try_to_timestamp(filing_date, 'MM/dd/yyyy')")) \
       .withColumn("issuance_date", F.expr("try_to_timestamp(issuance_date, 'yyyy-MM-dd')")) \
       .withColumn("expiration_date", F.expr("try_to_timestamp(expiration_date, 'MM/dd/yyyy')"))

# Replace nulls with standardized defaults
df = df.fillna({
    "borough": "UNKNOWN",
    "zip_code": "00000",
    "permit_status": "UNKNOWN",
    "work_type": "UNKNOWN",
    "residential": "UNKNOWN",
    "bldg_type": "UNKNOWN",
    "owner_s_business_name": "NOT PROVIDED",
    "owner_s_business_type": "NOT PROVIDED",
    "permittee_s_business_name": "NOT PROVIDED",
    "permittee_s_license_type": "NOT PROVIDED",
    "gis_nta_name": "UNKNOWN"
})

# Verify remaining nulls
remaining_nulls = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
print("Remaining nulls per column:")
for col, count in remaining_nulls.items():
    print(f"  {col}: {count}")

Remaining nulls per column:
  borough: 0
  zip_code: 0
  block: 9
  lot: 9
  job_type: 0
  permit_type: 0
  permit_status: 0
  filing_date: 29341
  issuance_date: 50000
  expiration_date: 41281
  work_type: 0
  residential: 0
  bldg_type: 0
  owner_s_business_name: 0
  owner_s_business_type: 0
  permittee_s_business_name: 0
  permittee_s_license_type: 0
  gis_latitude: 138
  gis_longitude: 138
  gis_nta_name: 0


#### Add Derived Columns
Calculated fields are added to enrich the dataset with 
analytical dimensions not present in the source data.

In [5]:
# Add derived columns
df = df.withColumn("permit_year", F.year("filing_date")) \
       .withColumn("permit_month", F.month("filing_date")) \
       .withColumn("days_to_expiration", 
           F.datediff("expiration_date", "filing_date")) \
       .withColumn("is_residential", 
           F.when(F.col("residential") == "YES", True).otherwise(False))

print("Derived columns added:")
df.select("filing_date", "permit_year", "permit_month", 
          "days_to_expiration", "is_residential").show(5)

Derived columns added:
+-------------------+-----------+------------+------------------+--------------+
|        filing_date|permit_year|permit_month|days_to_expiration|is_residential|
+-------------------+-----------+------------+------------------+--------------+
|2026-03-20 00:00:00|       2026|           3|               368|         false|
|2026-03-20 00:00:00|       2026|           3|               365|          true|
|2026-03-20 00:00:00|       2026|           3|              NULL|          true|
|2026-03-20 00:00:00|       2026|           3|              NULL|         false|
|2026-03-20 00:00:00|       2026|           3|              NULL|         false|
+-------------------+-----------+------------+------------------+--------------+
only showing top 5 rows


#### Save Cleaned Data
The transformed dataset is saved as a Parquet file — a columnar 
storage format optimized for analytical query performance and 
widely used in production data pipelines.

In [6]:
# Save cleaned data as Parquet
output_path = "/Users/catalinavalencia/Documents/ETL-ELT Project/data/cleaned_permits.parquet"

df.write.mode("overwrite").parquet(output_path)

print(f"Cleaned data saved to: {output_path}")
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")

[Stage 54:>                                                         (0 + 2) / 2]

Cleaned data saved to: /Users/catalinavalencia/Documents/ETL-ELT Project/data/cleaned_permits.parquet
Total rows: 50000
Total columns: 24
